# Combined JEDI + OmniParser smoke (single runtime)

Runs BOTH JEDI grounding (via vllm) and OmniParser parsing (via Florence-2) in one Colab runtime.

**This is the "can we have it all" version.** It requires vllm 0.8.2 + transformers 4.49.0, which earlier crashed on JEDI load. If it works here, we can drop the split-notebook architecture.

**Fallback if this crashes:** use the split notebooks ([`jedi_smoke.ipynb`](./jedi_smoke.ipynb) + [`omniparser_smoke.ipynb`](./omniparser_smoke.ipynb)) with serialized JSON passes.

**Prereqs:** Colab Pro A100, HF read token, OmniParser weights cached at `/content/drive/MyDrive/omniparser-weights/` (pull via jedi_smoke cell 3 if missing).

## Cell 1 — Bootstrap: Drive, repo, pinned deps

Pin rationale:
- `transformers==4.49.0` — Florence-2 trust_remote_code needs this
- `vllm==0.8.2` — highest vllm accepting transformers 4.49 that has Qwen2.5-VL
- `Pillow<11` — transformers 4.49 imports `_Ink` from `PIL._typing` (removed in Pillow 11)

**After first run, RESTART RUNTIME** (once, because of `--force-reinstall`), then re-run from cell 1.

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/isaacau502/GUI-grounded-gen'
REPO_DIR = '/content/GUI-grounded-gen'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    head = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD']).decode().strip()
    print(f'Cloned fresh @ {head}')
else:
    before = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD']).decode().strip()
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    after = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD']).decode().strip()
    if before == after:
        print(f'No new commits (HEAD: {after})')
    else:
        print(f'Pulled {before} -> {after}:')
        print(subprocess.check_output(['git', '-C', REPO_DIR, 'log', '--oneline', f'{before}..{after}']).decode().strip())

# Combined install: pinned transformers + vllm + Pillow + all OmniParser deps
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers==4.49.0',
                'vllm==0.8.2',
                'Pillow<11',
                'qwen-vl-utils',
                'huggingface_hub',
                'hf_transfer',
                'safetensors',
                'ultralytics',
                'easyocr',
                'matplotlib',
                '--force-reinstall'], check=True)

print('Bootstrap complete.')
print('If first run: RESTART RUNTIME now, then re-run cells 1+.')

## Cell 2 — HF token

In [ ]:
import os, getpass
from huggingface_hub import HfApi

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF token: ')

me = HfApi().whoami()
print(f"HF authed as: {me.get('name', '?')}")

## Cell 3 — Ensure weights on Drive (idempotent)

In [ ]:
import os, shutil
from pathlib import Path
from huggingface_hub import snapshot_download

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

def fetch(repo_id, local_dir, drive_dir, sentinel='config.json'):
    if any(os.path.exists(os.path.join(drive_dir, sub, sentinel))
           for sub in ['.', 'icon_caption', 'icon_caption_florence']):
        print(f'{repo_id}: already at {drive_dir}, skipping.')
        return
    print(f'{repo_id}: downloading to {local_dir}...')
    snapshot_download(repo_id=repo_id, local_dir=local_dir,
                      resume_download=True, token=os.environ['HF_TOKEN'])
    shutil.copytree(local_dir, drive_dir, dirs_exist_ok=True)
    print(f'{repo_id}: done.')

fetch('xlangai/Jedi-7B-1080p',
      '/content/jedi-weights',
      '/content/drive/MyDrive/jedi-weights')

fetch('microsoft/OmniParser-v2.0',
      '/content/omniparser-weights',
      '/content/drive/MyDrive/omniparser-weights')

# Rename icon_caption -> icon_caption_florence if needed
src = Path('/content/drive/MyDrive/omniparser-weights/icon_caption')
dst = Path('/content/drive/MyDrive/omniparser-weights/icon_caption_florence')
if src.exists() and not dst.exists():
    src.rename(dst)
    print(f'Renamed {src.name} -> {dst.name}')

## Cell 4 — Load JEDI via vllm 0.8.2

**This is the canary cell.** If vllm 0.8.2 + transformers 4.49 combo can load JEDI here, the combined runtime works. If it crashes with `Got fatal signal from worker processes`, fall back to split notebooks.

In [ ]:
import torch
from vllm import LLM, SamplingParams

gpu = torch.cuda.get_device_name(0)
dtype = 'bfloat16' if ('A100' in gpu or 'H100' in gpu) else 'float16'
print(f'GPU: {gpu}  dtype: {dtype}')

llm = LLM(
    model='/content/drive/MyDrive/jedi-weights',
    dtype=dtype,
    gpu_memory_utilization=0.5,   # reduced from 0.9: leave room for Florence-2
    max_model_len=8192,
    limit_mm_per_prompt={'image': 1},
)
print('JEDI loaded.')

## Cell 5 — JEDI smoke query on sample 1

Loads `grounding/jedi.py`, hot-swaps in the already-loaded `llm`, runs one click query. Same verification as `jedi_smoke.ipynb`.

In [ ]:
import sys
if '/content/GUI-grounded-gen' not in sys.path:
    sys.path.insert(0, '/content/GUI-grounded-gen')

from grounding.jedi import JEDI
from grounding.prompts import CLICK_ELEMENT, format_query

jedi = JEDI.__new__(JEDI)
jedi.llm = llm

SCREENSHOT_PATH = '/content/drive/MyDrive/samples/1.png'
ISSUE_TYPE = "a button labeled 'Upload Photo'"

instruction = format_query(CLICK_ELEMENT, issue_type=ISSUE_TYPE)
result = jedi.query(SCREENSHOT_PATH, instruction)

print(f"Raw:          {result['raw_output']}")
print(f"Point (orig): {result['point']}")
print(f"Parsed OK:    {result['parse_success']}")
# Expected: x ~ 685, y ~ 308 (click lands on Upload Photo button)

## Cell 6 — Free JEDI VRAM before loading OmniParser

vllm's worker processes hold ~14GB of VRAM for JEDI weights. We need to free them before loading Florence-2/YOLO. Delete the LLM handle + force garbage collection + empty CUDA cache.

In [ ]:
import gc
del llm, jedi
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# Verify VRAM is freed
!nvidia-smi | head -15

## Cell 7 — Run OmniParser prod (3 variants × 5 samples)

In [ ]:
!cd /content/GUI-grounded-gen && git pull
!python /content/GUI-grounded-gen/scripts/prod_omniparser.py

## Cell 8 — Notes

### If cell 4 (JEDI load) crashes with `Got fatal signal from worker processes`

vllm 0.8.2 has known issues loading Qwen2.5-VL in some configurations. Fall back to split notebooks:
- `jedi_smoke.ipynb` for JEDI (vllm latest + transformers latest)
- `omniparser_smoke.ipynb` for OmniParser (transformers 4.49 + no vllm)
- Pipeline results serialize through Drive JSON

### If everything succeeds

Drop the split-notebook architecture. This combined notebook becomes the primary entry point. Both grounding signals live in one Colab runtime; downstream pipeline code can call JEDI + OmniParser without cross-notebook serialization.